In [72]:
%pip install fastapi uvicorn python-multipart requests pydantic

Note: you may need to restart the kernel to use updated packages.


In [73]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

BACKEND_PATH = PROJECT_ROOT / "backend"
API_PATH = BACKEND_PATH / "api"
ROUTES_PATH = API_PATH / "routes"
SERVICES_PATH = BACKEND_PATH / "services"
SCHEMAS_PATH = BACKEND_PATH / "schemas"

for folder in [
    BACKEND_PATH,
    API_PATH,
    ROUTES_PATH,
    SERVICES_PATH,
    SCHEMAS_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Backend path:", BACKEND_PATH)

Project root: /Users/rizwansalim/Desktop/SkillBridgeByZequa
Backend path: /Users/rizwansalim/Desktop/SkillBridgeByZequa/backend


In [74]:
required_modules = [
    PROJECT_ROOT / "data/processed/cv_builder/updated_candidate.json",
    PROJECT_ROOT / "data/processed/candidates/complete_candidate_career_report.json",
    PROJECT_ROOT / "data/processed/career_mentor/mentor_candidate_profile.json",
    PROJECT_ROOT / "data/processed/career_mentor/career_faiss.index"
]

print("SKILLBRIDGE BACKEND INPUT VALIDATION")
print("-" * 45)

for path in required_modules:
    print("FOUND" if path.exists() else "MISSING", "|", path.name)

SKILLBRIDGE BACKEND INPUT VALIDATION
---------------------------------------------
FOUND | updated_candidate.json
FOUND | complete_candidate_career_report.json
FOUND | mentor_candidate_profile.json
FOUND | career_faiss.index


In [75]:
from pathlib import Path

backend_folders = [
    "api/routes",
    "services",
    "schemas",
    "utils"
]

for folder in backend_folders:
    path = BACKEND_PATH / folder
    path.mkdir(parents=True, exist_ok=True)

    print("CREATED:", path.relative_to(PROJECT_ROOT))

CREATED: backend/api/routes
CREATED: backend/services
CREATED: backend/schemas
CREATED: backend/utils


In [76]:
package_files = [
    "api/__init__.py",
    "api/routes/__init__.py",
    "services/__init__.py",
    "schemas/__init__.py",
    "utils/__init__.py"
]

for file_name in package_files:
    file_path = BACKEND_PATH / file_name
    file_path.touch(exist_ok=True)

print("Python package files created!")

Python package files created!


In [77]:
config_code = '''
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent

APP_NAME = "SkillBridge AI Career Platform"
APP_VERSION = "1.0.0"

DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models"

CV_PATH = DATA_PATH / "cv_builder"
CAREER_PATH = DATA_PATH / "candidates"
MENTOR_PATH = DATA_PATH / "career_mentor"

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:3b"
'''

config_path = BACKEND_PATH / "config.py"
config_path.write_text(config_code.strip(), encoding="utf-8")

print("Configuration saved:", config_path)
print("File exists:", config_path.exists())

Configuration saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/backend/config.py
File exists: True


In [78]:
for folder in backend_folders:
    path = BACKEND_PATH / folder
    print("FOUND" if path.exists() else "MISSING", "|", folder)

print(
    "FOUND" if config_path.exists() else "MISSING",
    "| config.py"
)

FOUND | api/routes
FOUND | services
FOUND | schemas
FOUND | utils
FOUND | config.py


In [79]:
main_code = '''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="SkillBridge AI Career Platform",
    description="AI-powered career development and job matching platform",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/")
def home():
    return {
        "message": "Welcome to SkillBridge AI Career Platform",
        "status": "Running",
        "version": "1.0.0"
    }

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "application": "SkillBridge"
    }
'''

main_path = BACKEND_PATH / "main.py"
main_path.write_text(main_code.strip(), encoding="utf-8")

print("FastAPI main file created:", main_path)

FastAPI main file created: /Users/rizwansalim/Desktop/SkillBridgeByZequa/backend/main.py


In [80]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)

print("FastAPI application imported successfully!")

FastAPI application imported successfully!


In [81]:
home_response = client.get("/")

print("Status code:", home_response.status_code)
print("Response:", home_response.json())

Status code: 200
Response: {'message': 'Welcome to SkillBridge AI Career Platform', 'status': 'Running', 'version': '1.0.0'}


In [82]:
health_response = client.get("/health")

print("Status code:", health_response.status_code)
print("Response:", health_response.json())

Status code: 200
Response: {'status': 'healthy', 'application': 'SkillBridge'}


In [83]:
candidate_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import CV_PATH

router = APIRouter(prefix="/candidate", tags=["Candidate"])

RESUME_LIBRARY = CV_PATH / "resume_library"

@router.get("/resumes")
def list_candidate_resumes():
    RESUME_LIBRARY.mkdir(parents=True, exist_ok=True)
    profiles = []
    for path in sorted(RESUME_LIBRARY.glob("*.json")):
        try:
            with open(path, "r", encoding="utf-8") as file:
                profile = json.load(file)
            personal = profile.get("personal_details", {})
            profiles.append({
                "resume_id": path.stem,
                "candidate_name": personal.get("full_name", "Unknown"),
                "email": personal.get("email", ""),
                "file_name": path.name
            })
        except (OSError, json.JSONDecodeError):
            continue
    return {"status": "success", "resumes": profiles, "total": len(profiles)}

@router.get("/profile")
def get_candidate_profile(resume_id: str = None):
    RESUME_LIBRARY.mkdir(parents=True, exist_ok=True)

    if resume_id:
        profile_path = RESUME_LIBRARY / f"{resume_id}.json"
        if not profile_path.exists():
            raise HTTPException(status_code=404, detail="Requested resume not found")
    else:
        saved_profiles = sorted(
            RESUME_LIBRARY.glob("*.json"),
            key=lambda path: path.stat().st_mtime,
            reverse=True
        )
        profile_path = saved_profiles[0] if saved_profiles else CV_PATH / "updated_candidate.json"

    if not profile_path.exists():
        raise HTTPException(status_code=404, detail="Candidate profile not found")

    try:
        with open(profile_path, "r", encoding="utf-8") as file:
            profile = json.load(file)
    except json.JSONDecodeError:
        raise HTTPException(status_code=500, detail="Candidate profile JSON is invalid")

    return {
        "status": "success",
        "resume_id": profile_path.stem,
        "candidate": profile
    }
'''

candidate_route_path = ROUTES_PATH / "candidate.py"
candidate_route_path.write_text(candidate_route_code.strip(), encoding="utf-8")
print("Resume-aware candidate routes created!")


Resume-aware candidate routes created!


In [84]:
main_code = main_path.read_text(encoding="utf-8")

if "candidate.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import candidate"
    )

    main_code += "\napp.include_router(candidate.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Candidate API registered!")

Candidate API registered!


In [85]:
import importlib
import backend.main

importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/candidate/profile")

print("Status code:", response.status_code)
print("Response status:", response.json().get("status"))

Status code: 200
Response status: success


In [86]:
if response.status_code == 200 and isinstance(response.json(), dict) and "candidate" in response.json():
    candidate_result = response.json()["candidate"]
    print("Candidate Profile")
    print("-" * 35)
    print("Available sections:", list(candidate_result.keys()))
    print("Personal details:", candidate_result.get("personal_details", {}))
    print("Skills:", len(candidate_result.get("skills", [])))
    print("Projects:", len(candidate_result.get("projects", [])))
else:
    print("Candidate profile is not available yet.")
    print("HTTP status:", response.status_code)
    try:
        print("API detail:", response.json())
    except ValueError:
        print("API returned a non-JSON response.")


Candidate Profile
-----------------------------------
Available sections: ['personal_details', 'education', 'experience', 'projects', 'skills', 'certifications', 'languages']
Personal details: {'full_name': '', 'email': '', 'phone': '', 'location': '', 'linkedin': '', 'portfolio': '', 'professional_summary': ''}
Skills: 5
Projects: 0


In [87]:
if response.status_code == 200 and isinstance(response.json(), dict) and "candidate" in response.json():
    candidate_result = response.json()["candidate"]
    print("Candidate Profile")
    print("-" * 35)
    print("Available sections:", list(candidate_result.keys()))
    print("Personal details:", candidate_result.get("personal_details", {}))
    print("Skills:", len(candidate_result.get("skills", [])))
    print("Projects:", len(candidate_result.get("projects", [])))
else:
    print("Candidate profile is not available yet.")
    print("HTTP status:", response.status_code)
    try:
        print("API detail:", response.json())
    except ValueError:
        print("API returned a non-JSON response.")


Candidate Profile
-----------------------------------
Available sections: ['personal_details', 'education', 'experience', 'projects', 'skills', 'certifications', 'languages']
Personal details: {'full_name': '', 'email': '', 'phone': '', 'location': '', 'linkedin': '', 'portfolio': '', 'professional_summary': ''}
Skills: 5
Projects: 0


In [88]:
UPLOAD_PATH = PROJECT_ROOT / "data" / "processed" / "uploaded_resumes"

UPLOAD_PATH.mkdir(parents=True, exist_ok=True)

print("Upload directory:", UPLOAD_PATH)

Upload directory: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/uploaded_resumes


In [89]:
candidate_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import CV_PATH

router = APIRouter(prefix="/candidate", tags=["Candidate"])

RESUME_LIBRARY = CV_PATH / "resume_library"

@router.get("/resumes")
def list_candidate_resumes():
    RESUME_LIBRARY.mkdir(parents=True, exist_ok=True)
    profiles = []
    for path in sorted(RESUME_LIBRARY.glob("*.json")):
        try:
            with open(path, "r", encoding="utf-8") as file:
                profile = json.load(file)
            personal = profile.get("personal_details", {})
            profiles.append({
                "resume_id": path.stem,
                "candidate_name": personal.get("full_name", "Unknown"),
                "email": personal.get("email", ""),
                "file_name": path.name
            })
        except (OSError, json.JSONDecodeError):
            continue
    return {"status": "success", "resumes": profiles, "total": len(profiles)}

@router.get("/profile")
def get_candidate_profile(resume_id: str = None):
    RESUME_LIBRARY.mkdir(parents=True, exist_ok=True)

    if resume_id:
        profile_path = RESUME_LIBRARY / f"{resume_id}.json"
        if not profile_path.exists():
            raise HTTPException(status_code=404, detail="Requested resume not found")
    else:
        saved_profiles = sorted(
            RESUME_LIBRARY.glob("*.json"),
            key=lambda path: path.stat().st_mtime,
            reverse=True
        )
        profile_path = saved_profiles[0] if saved_profiles else CV_PATH / "updated_candidate.json"

    if not profile_path.exists():
        raise HTTPException(status_code=404, detail="Candidate profile not found")

    try:
        with open(profile_path, "r", encoding="utf-8") as file:
            profile = json.load(file)
    except json.JSONDecodeError:
        raise HTTPException(status_code=500, detail="Candidate profile JSON is invalid")

    return {
        "status": "success",
        "resume_id": profile_path.stem,
        "candidate": profile
    }
'''

candidate_route_path = ROUTES_PATH / "candidate.py"
candidate_route_path.write_text(candidate_route_code.strip(), encoding="utf-8")
print("Resume-aware candidate routes created!")


Resume-aware candidate routes created!


In [90]:
main_code = main_path.read_text(encoding="utf-8")

if "candidate.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import candidate"
    )

    main_code += "\napp.include_router(candidate.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Candidate API registered!")

Candidate API registered!


In [91]:
import importlib
import backend.main

importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/candidate/profile")

print("Status code:", response.status_code)
print("Response status:", response.json().get("status"))

Status code: 200
Response status: success


In [92]:
if response.status_code == 200 and isinstance(response.json(), dict) and "candidate" in response.json():
    candidate_result = response.json()["candidate"]
    print("Candidate Profile")
    print("-" * 35)
    print("Available sections:", list(candidate_result.keys()))
    print("Personal details:", candidate_result.get("personal_details", {}))
    print("Skills:", len(candidate_result.get("skills", [])))
    print("Projects:", len(candidate_result.get("projects", [])))
else:
    print("Candidate profile is not available yet.")
    print("HTTP status:", response.status_code)
    try:
        print("API detail:", response.json())
    except ValueError:
        print("API returned a non-JSON response.")


Candidate Profile
-----------------------------------
Available sections: ['personal_details', 'education', 'experience', 'projects', 'skills', 'certifications', 'languages']
Personal details: {'full_name': '', 'email': '', 'phone': '', 'location': '', 'linkedin': '', 'portfolio': '', 'professional_summary': ''}
Skills: 5
Projects: 0


In [93]:
UPLOAD_PATH = PROJECT_ROOT / "data" / "processed" / "uploaded_resumes"

UPLOAD_PATH.mkdir(parents=True, exist_ok=True)

print("Upload directory:", UPLOAD_PATH)

Upload directory: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/uploaded_resumes


In [94]:
resume_route_code = '''
from pathlib import Path
from uuid import uuid4
from fastapi import APIRouter, UploadFile, File, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/resume",
    tags=["Resume"]
)

UPLOAD_PATH = PROJECT_ROOT / "data" / "processed" / "uploaded_resumes"
UPLOAD_PATH.mkdir(parents=True, exist_ok=True)

@router.post("/upload")
async def upload_resume(file: UploadFile = File(...)):

    if file.content_type != "application/pdf":
        raise HTTPException(
            status_code=400,
            detail="Only PDF files are allowed"
        )

    if not file.filename or not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Invalid PDF filename"
        )

    content = await file.read()

    if not content.startswith(b"%PDF-"):
        raise HTTPException(
            status_code=400,
            detail="Uploaded file is not a valid PDF"
        )

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(
            status_code=413,
            detail="File size exceeds 10 MB"
        )

    saved_name = f"{uuid4().hex}.pdf"
    save_path = UPLOAD_PATH / saved_name

    save_path.write_bytes(content)

    return {
        "status": "success",
        "original_filename": file.filename,
        "saved_filename": saved_name,
        "size_bytes": len(content),
        "message": "Resume uploaded successfully"
    }
'''

resume_route_path = ROUTES_PATH / "resume.py"

resume_route_path.write_text(
    resume_route_code.strip(),
    encoding="utf-8"
)

print("Resume upload route created!")

Resume upload route created!


In [95]:
main_code = main_path.read_text(encoding="utf-8")

if "resume.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import resume"
    )

    main_code += "\napp.include_router(resume.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Resume upload API registered!")

Resume upload API registered!


In [96]:
import importlib
import backend.main

importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.post(
    "/resume/upload",
    files={
        "file": (
            "test_resume.pdf",
            b"%PDF-1.4\nSkillBridge test PDF",
            "application/pdf"
        )
    }
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'success', 'original_filename': 'test_resume.pdf', 'saved_filename': '1b149979cd474bf9acc9979230ce6c5f.pdf', 'size_bytes': 29, 'message': 'Resume uploaded successfully'}


In [97]:
saved_filename = response.json().get("saved_filename")

saved_file = UPLOAD_PATH / saved_filename

print("Uploaded file exists:", saved_file.exists())
print("File size:", saved_file.stat().st_size, "bytes")

Uploaded file exists: True
File size: 29 bytes


In [98]:
parser_route_code = '''
import fitz
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/resume",
    tags=["Resume"]
)

UPLOAD_PATH = PROJECT_ROOT / "data" / "processed" / "uploaded_resumes"

@router.get("/parse/{filename}")
def parse_resume(filename: str):

    if not filename.endswith(".pdf") or "/" in filename or "\\\\" in filename:
        raise HTTPException(
            status_code=400,
            detail="Invalid filename"
        )

    file_path = UPLOAD_PATH / filename

    if not file_path.exists():
        raise HTTPException(
            status_code=404,
            detail="Resume not found"
        )

    try:
        with fitz.open(file_path) as document:
            extracted_text = "\\n".join(
                page.get_text() for page in document
            )

        if not extracted_text.strip():
            raise HTTPException(
                status_code=422,
                detail="No readable text found in PDF"
            )

        return {
            "status": "success",
            "filename": filename,
            "pages": len(document),
            "characters_extracted": len(extracted_text),
            "extracted_text": extracted_text
        }

    except HTTPException:
        raise

    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"Resume parsing failed: {str(error)}"
        )
'''

parser_route_path = ROUTES_PATH / "resume_parser.py"

parser_route_path.write_text(
    parser_route_code.strip(),
    encoding="utf-8"
)

print("Resume parsing route created!")

Resume parsing route created!


In [99]:
main_code = main_path.read_text(encoding="utf-8")

if "resume_parser.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import resume_parser"
    )

    main_code += "\napp.include_router(resume_parser.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Resume parsing API registered!")

Resume parsing API registered!


In [100]:
import importlib
import backend.main

importlib.reload(backend.main)

client = TestClient(backend.main.app)

print("Resume parsing API loaded!")

Resume parsing API loaded!


In [143]:
import fitz

test_pdf_path = UPLOAD_PATH / "skillbridge_test_resume.pdf"

doc = fitz.open()

page = doc.new_page()

page.insert_text(
    (50, 60),
    "Rizwan Salim\n"
    "Data Science Fresher\n\n"
    "SKILLS\n"
    "Python, SQL, Machine Learning, Pandas\n\n"
    "PROJECTS\n"
    "SkillBridge AI Career Platform\n"
    "Sales Prediction System\n\n"
    "EDUCATION\n"
    "B.Tech"
)

doc.save(test_pdf_path)
doc.close()

print("Valid PDF created:", test_pdf_path.exists())

Valid PDF created: True


In [144]:
parser_route_code = '''
import fitz
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(prefix="/resume", tags=["Resume"])

UPLOAD_PATH = PROJECT_ROOT / "data" / "processed" / "uploaded_resumes"

@router.get("/parse/{filename}")
def parse_resume(filename: str):

    if (
        not filename.endswith(".pdf")
        or "/" in filename
        or "\\\\" in filename
    ):
        raise HTTPException(status_code=400, detail="Invalid filename")

    file_path = UPLOAD_PATH / filename

    if not file_path.exists():
        raise HTTPException(status_code=404, detail="Resume not found")

    try:
        with fitz.open(file_path) as document:
            page_count = len(document)
            extracted_text = "\\n".join(
                page.get_text() for page in document
            )

        if not extracted_text.strip():
            raise HTTPException(
                status_code=422,
                detail="No readable text found in PDF"
            )

        return {
            "status": "success",
            "filename": filename,
            "pages": page_count,
            "characters_extracted": len(extracted_text),
            "extracted_text": extracted_text
        }

    except HTTPException:
        raise

    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"Resume parsing failed: {str(error)}"
        )
'''

parser_route_path.write_text(
    parser_route_code.strip(),
    encoding="utf-8"
)

print("Resume parser updated successfully!")

Resume parser updated successfully!


In [145]:
import importlib
import backend.api.routes.resume_parser
import backend.main

importlib.reload(backend.api.routes.resume_parser)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

parse_response = client.get(
    "/resume/parse/skillbridge_test_resume.pdf"
)

print("Status code:", parse_response.status_code)
print("Response:", parse_response.json())

Status code: 200
Response: {'status': 'success', 'filename': 'skillbridge_test_resume.pdf', 'pages': 1, 'characters_extracted': 160, 'extracted_text': 'Rizwan Salim\nData Science Fresher\nSKILLS\nPython, SQL, Machine Learning, Pandas\nPROJECTS\nSkillBridge AI Career Platform\nSales Prediction System\nEDUCATION\nB.Tech\n'}


In [146]:
career_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/career",
    tags=["Career Recommendations"]
)

REPORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "candidates" / "complete_candidate_career_report.json"
)

@router.get("/recommendations")
def get_career_recommendations():

    if not REPORT_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Career report not found"
        )

    with open(REPORT_PATH, "r", encoding="utf-8") as file:
        report = json.load(file)

    recommendations = report.get("career_recommendations", [])

    return {
        "status": "success",
        "candidate": report.get("candidate_name"),
        "recommendations": recommendations,
        "total_recommendations": len(recommendations)
    }
'''

career_route_path = ROUTES_PATH / "career.py"

career_route_path.write_text(
    career_route_code.strip(),
    encoding="utf-8"
)

print("Career recommendation route created!")

Career recommendation route created!


In [147]:
main_code = main_path.read_text(encoding="utf-8")

if "career.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import career"
    )

    main_code += "\napp.include_router(career.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Career API registered!")

Career API registered!


In [148]:
import importlib
import backend.api.routes.career
import backend.main

importlib.reload(backend.api.routes.career)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/career/recommendations")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'success', 'candidate': None, 'recommendations': [{'Career': 'IoT developer', 'Match Score': 9.09, 'Matched Skills': ['computer science', 'machine learning'], 'Missing Skills': ['algorithms', 'analyse big data', 'computer technology', 'design information system', 'develop ict workflow', 'digital data processing', 'establish data processes', 'hardware testing methods', 'ict architectural frameworks', 'ict software specifications', 'ict system programming', 'internet of things', 'mechatronics', 'mobile device software frameworks', 'perform dimensionality reduction', 'perform online data analysis', 'principles of artificial intelligence', 'task algorithmisation', 'use data processing techniques', 'utilise machine learning'], 'Total Required Skills': 22}, {'Career': 'battery simulation engineer', 'Match Score': 7.69, 'Matched Skills': ['computer science'], 'Missing Skills': ['algorithms', 'battery design', 'computer programming', 'develop predictive mo

In [149]:
skill_gap_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/career",
    tags=["Skill Gap Analysis"]
)

REPORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "candidates" / "complete_candidate_career_report.json"
)

@router.get("/skill-gap")
def get_skill_gap():

    if not REPORT_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Career report not found"
        )

    with open(REPORT_PATH, "r", encoding="utf-8") as file:
        report = json.load(file)

    skill_gap = report.get("skill_gap_analysis", [])

    return {
        "status": "success",
        "candidate": report.get("candidate_name", "Unknown"),
        "skill_gap_analysis": skill_gap
    }
'''

skill_gap_path = ROUTES_PATH / "skill_gap.py"

skill_gap_path.write_text(
    skill_gap_route_code.strip(),
    encoding="utf-8"
)

print("Skill gap route created!")

Skill gap route created!


In [150]:
main_code = main_path.read_text(encoding="utf-8")

if "skill_gap.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import skill_gap"
    )

    main_code += "\napp.include_router(skill_gap.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Skill gap API registered!")

Skill gap API registered!


In [151]:
import importlib
import backend.api.routes.skill_gap
import backend.main

importlib.reload(backend.api.routes.skill_gap)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/career/skill-gap")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'success', 'candidate': 'Unknown', 'skill_gap_analysis': {'Target Career': 'IoT developer', 'Existing Skills': ['computer science', 'machine learning'], 'Missing Skills': ['algorithms', 'analyse big data', 'computer technology', 'design information system', 'develop ict workflow', 'digital data processing', 'establish data processes', 'hardware testing methods', 'ict architectural frameworks', 'ict software specifications', 'ict system programming', 'internet of things', 'mechatronics', 'mobile device software frameworks', 'perform dimensionality reduction', 'perform online data analysis', 'principles of artificial intelligence', 'task algorithmisation', 'use data processing techniques', 'utilise machine learning'], 'Total Required Skills': 22, 'Matched Skills Count': 2, 'Missing Skills Count': 20, 'Skill Coverage Percentage': 9.09}}


In [110]:
if response.status_code == 200:

    gap_data = response.json()

    print("Candidate:", gap_data["candidate"])
    print("Skill-gap analysis:")

    print(gap_data["skill_gap_analysis"])

Candidate: Unknown
Skill-gap analysis:
{'Target Career': 'IoT developer', 'Existing Skills': ['computer science', 'machine learning'], 'Missing Skills': ['algorithms', 'analyse big data', 'computer technology', 'design information system', 'develop ict workflow', 'digital data processing', 'establish data processes', 'hardware testing methods', 'ict architectural frameworks', 'ict software specifications', 'ict system programming', 'internet of things', 'mechatronics', 'mobile device software frameworks', 'perform dimensionality reduction', 'perform online data analysis', 'principles of artificial intelligence', 'task algorithmisation', 'use data processing techniques', 'utilise machine learning'], 'Total Required Skills': 22, 'Matched Skills Count': 2, 'Missing Skills Count': 20, 'Skill Coverage Percentage': 9.09}


In [111]:
CV_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cv_builder"

print("CV Builder Files")
print("-" * 35)

for file in CV_OUTPUT_PATH.iterdir():
    if file.is_file():
        print(file.name)

CV Builder Files
-----------------------------------
updated_candidate.json
resume_library_index.json
skillbridge_generated_cv.docx
cv_templates.json
resume_template.json
generated_cv.json
skillbridge_generated_cv.pdf
resume_score_report.json


In [112]:
generated_pdf = CV_OUTPUT_PATH / "skillbridge_generated_cv.pdf"

print("PDF exists:", generated_pdf.exists())

if generated_pdf.exists():
    print("PDF size:", generated_pdf.stat().st_size, "bytes")

PDF exists: True
PDF size: 1797 bytes


In [113]:
CV_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cv_builder"

pdf_files = list(CV_OUTPUT_PATH.glob("*.pdf"))

print("Available PDF files:")

for file in pdf_files:
    print(file.name)

Available PDF files:
skillbridge_generated_cv.pdf


In [114]:
cv_route_code = '''
from fastapi import APIRouter, HTTPException
from fastapi.responses import FileResponse
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/cv",
    tags=["CV Builder"]
)

CV_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "cv_builder" / "skillbridge_generated_cv.pdf"
)

@router.get("/download")
def download_cv():

    if not CV_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Generated CV not found"
        )

    return FileResponse(
        path=CV_PATH,
        media_type="application/pdf",
        filename="SkillBridge_Generated_CV.pdf"
    )
'''

cv_route_path = ROUTES_PATH / "cv.py"

cv_route_path.write_text(
    cv_route_code.strip(),
    encoding="utf-8"
)

print("CV download route created!")

CV download route created!


In [115]:
main_code = main_path.read_text(encoding="utf-8")

if "cv.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import cv"
    )

    main_code += "\napp.include_router(cv.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("CV API registered!")

CV API registered!


In [116]:
import importlib
import backend.api.routes.cv
import backend.main

importlib.reload(backend.api.routes.cv)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/cv/download")

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))
print("PDF size:", len(response.content), "bytes")

Status code: 200
Content type: application/pdf
PDF size: 1797 bytes


In [117]:
job_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/jobs",
    tags=["Job Matching"]
)

REPORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "job_analyzer" / "complete_job_analysis_report.json"
)

@router.get("/matches")
def get_job_matches():

    if not REPORT_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Job analysis report not found"
        )

    with open(REPORT_PATH, "r", encoding="utf-8") as file:
        report = json.load(file)

    return {
        "status": "success",
        "candidate": report.get("candidate_name", "Unknown"),
        "job_analysis": report
    }
'''

job_route_path = ROUTES_PATH / "jobs.py"

job_route_path.write_text(
    job_route_code.strip(),
    encoding="utf-8"
)

print("Job matching route created!")

Job matching route created!


In [118]:
main_code = main_path.read_text(encoding="utf-8")

if "jobs.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import jobs"
    )

    main_code += "\napp.include_router(jobs.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Job matching API registered!")

Job matching API registered!


In [119]:
import importlib
import backend.api.routes.jobs
import backend.main

importlib.reload(backend.api.routes.jobs)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/jobs/matches")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'success', 'candidate': 'Unknown', 'job_analysis': {'candidate': {'name': 'Candidate', 'email': '', 'skills': ['computer science', 'deep learning', 'machine learning', 'Science', 'SQL']}, 'total_jobs_analyzed': 4, 'job_recommendations': [{'rank': 1, 'job_id': 'JOB001', 'job_title': 'Data Scientist', 'company': 'ABC Technologies', 'match_score': 70.0, 'exact_coverage': 100.0, 'semantic_coverage': 0.0, 'missing_skills': [], 'suggested_related_skills': []}, {'rank': 2, 'job_id': 'JOB002', 'job_title': 'Machine Learning Engineer', 'company': 'XYZ Solutions', 'match_score': 46.67, 'exact_coverage': 66.67, 'semantic_coverage': 0.0, 'missing_skills': ['cloud technologies'], 'suggested_related_skills': []}, {'rank': 3, 'job_id': 'JOB003', 'job_title': 'Data Analyst', 'company': 'TechVision', 'match_score': 35.0, 'exact_coverage': 50.0, 'semantic_coverage': 0.0, 'missing_skills': ['business analysis'], 'suggested_related_skills': []}, {'rank': 4, 'job_id': 

In [120]:
if response.status_code == 200:

    job_data = response.json()

    print("Candidate:", job_data["candidate"])
    print("Available report sections:",
          list(job_data["job_analysis"].keys()))

Candidate: Unknown
Available report sections: ['candidate', 'total_jobs_analyzed', 'job_recommendations', 'analysis_summary']


In [121]:
INTERVIEW_PATH = (
    PROJECT_ROOT / "data" / "processed" / "mock_interview"
)

print("Interview files:")

if INTERVIEW_PATH.exists():
    for file in INTERVIEW_PATH.iterdir():
        if file.is_file():
            print(file.name)
else:
    print("Interview folder not found.")

Interview files:
interview_performance_report.json
complete_interview_report.json
technical_question_bank.json
hr_question_bank.json
generated_interview.json
interview_candidate_profile.json
interview_feedback.csv
candidate_interview_answers.json


In [122]:
interview_route_code = '''
import json
from fastapi import APIRouter, HTTPException
from backend.config import PROJECT_ROOT

router = APIRouter(
    prefix="/interview",
    tags=["Mock Interview"]
)

REPORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "mock_interview" / "complete_interview_report.json"
)

@router.get("/report")
def get_interview_report():

    if not REPORT_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Interview report not found"
        )

    with open(REPORT_PATH, "r", encoding="utf-8") as file:
        report = json.load(file)

    return {
        "status": "success",
        "interview_report": report
    }
'''

interview_route_path = ROUTES_PATH / "interview.py"

interview_route_path.write_text(
    interview_route_code.strip(),
    encoding="utf-8"
)

print("Interview route created!")

Interview route created!


In [123]:
main_code = main_path.read_text(encoding="utf-8")

if "interview.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import interview"
    )

    main_code += "\napp.include_router(interview.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Interview API registered!")

Interview API registered!


In [124]:
import importlib
import backend.api.routes.interview
import backend.main

importlib.reload(backend.api.routes.interview)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

response = client.get("/interview/report")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'success', 'interview_report': {'candidate': {'name': '', 'email': ''}, 'interview_details': {'target_role': 'Data Scientist', 'difficulty': 'Intermediate', 'date': '2026-11-26T00:14:20.974745'}, 'performance': {'total_questions': 7, 'average_score': 0.0, 'performance_level': 'Needs Improvement'}, 'category_performance': [{'Category': 'HR', 'Average Score': 0.0}, {'Category': 'Technical', 'Average Score': 0.0}], 'question_answers': [{'question_number': 1, 'category': 'HR', 'question': 'How do you handle deadlines?', 'answer': 'Data Science uses statistics, programming and machine learning to extract insights from data.', 'relevance_score': 0.0, 'answer_length': 13, 'feedback': 'Keyword and text similarity evaluation completed.'}, {'question_number': 2, 'category': 'Technical', 'question': 'What is the difference between precision and recall?', 'answer': 'Supervised learning uses labelled data, while unsupervised learning finds patterns in unlabelle

In [125]:
if response.status_code == 200:

    interview_data = response.json()["interview_report"]

    print("Available sections:", list(interview_data.keys()))

Available sections: ['candidate', 'interview_details', 'performance', 'category_performance', 'question_answers']


In [126]:
mentor_route_code = '''
import json
import requests
from fastapi import APIRouter, HTTPException
from pydantic import BaseModel
from backend.config import PROJECT_ROOT, OLLAMA_URL, OLLAMA_MODEL

router = APIRouter(
    prefix="/mentor",
    tags=["AI Career Mentor"]
)

PROFILE_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "career_mentor" / "mentor_candidate_profile.json"
)

class MentorQuestion(BaseModel):
    question: str

@router.post("/ask")
def ask_mentor(data: MentorQuestion):

    if not data.question.strip():
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty"
        )

    if not PROFILE_PATH.exists():
        raise HTTPException(
            status_code=404,
            detail="Mentor candidate profile not found"
        )

    with open(PROFILE_PATH, "r", encoding="utf-8") as file:
        candidate_profile = json.load(file)

    prompt = (
        "You are SkillBridge AI Career Mentor. "
        "Give practical, accurate and personalized career advice.\\n\\n"
        f"Candidate profile: {json.dumps(candidate_profile)}\\n\\n"
        f"Candidate question: {data.question}"
    )

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=180
        )

        response.raise_for_status()
        result = response.json()

        return {
            "status": "success",
            "question": data.question,
            "answer": result.get("response", "")
        }

    except requests.RequestException as error:
        raise HTTPException(
            status_code=503,
            detail=f"Career mentor service unavailable: {str(error)}"
        )
'''

mentor_route_path = ROUTES_PATH / "mentor.py"

mentor_route_path.write_text(
    mentor_route_code.strip(),
    encoding="utf-8"
)

print("Career mentor route created!")

Career mentor route created!


In [127]:
main_code = main_path.read_text(encoding="utf-8")

if "mentor.router" not in main_code:
    main_code = main_code.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom backend.api.routes import mentor"
    )

    main_code += "\napp.include_router(mentor.router)\n"

    main_path.write_text(main_code, encoding="utf-8")

print("Career mentor API registered!")

Career mentor API registered!


In [128]:
import importlib
import backend.api.routes.mentor
import backend.main

importlib.reload(backend.api.routes.mentor)
importlib.reload(backend.main)

client = TestClient(backend.main.app)

print("Career mentor API loaded!")

Career mentor API loaded!


In [129]:
response = client.post(
    "/mentor/ask",
    json={
        "question": "What skills should I learn to become a Data Scientist?"
    }
)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("AI Answer:")
    print(response.json()["answer"])
else:
    print(response.json())

Status code: 200
AI Answer:
Hello Candidate!

As a SkillBridge AI Career Mentor, I'd be delighted to help you on your career journey. Becoming a Data Scientist is an exciting career path that combines various skills, including computer science, mathematics, and domain expertise.

To become a Data Scientist, it's essential to develop a strong foundation in the following skills:

1. **Programming skills**:
	* Python: This is the go-to language for Data Science. Familiarize yourself with popular libraries like NumPy, pandas, and scikit-learn.
	* R: Also popular in the Data Science community, especially for statistical analysis.
	* SQL: Learn to work with relational databases using SQL.
2. **Data manipulation and analysis**:
	* Data cleaning and preprocessing
	* Data visualization (e.g., Matplotlib, Seaborn, Plotly)
	* Machine learning algorithms (e.g., regression, classification, clustering)
3. **Machine learning and deep learning**:
	* Supervised and unsupervised learning
	* Neural netwo

In [130]:
import importlib
import backend.main

importlib.reload(backend.main)

client = TestClient(backend.main.app)

api_tests = [
    ("Health API", "GET", "/health"),
    ("Home API", "GET", "/"),
    ("Candidate Profile API", "GET", "/candidate/profile"),
    ("Career Recommendations API", "GET", "/career/recommendations"),
    ("Skill Gap API", "GET", "/career/skill-gap"),
    ("CV Download API", "GET", "/cv/download"),
    ("Job Matching API", "GET", "/jobs/matches"),
    ("Mock Interview API", "GET", "/interview/report"),
]

print("SKILLBRIDGE API TESTING")
print("-" * 55)

for name, method, endpoint in api_tests:
    try:
        result = client.get(endpoint)

        print(
            f"{'PASS' if result.status_code == 200 else 'CHECK'} | "
            f"{name} | HTTP {result.status_code}"
        )

    except Exception as error:
        print(f"ERROR | {name} | {error}")

SKILLBRIDGE API TESTING
-------------------------------------------------------
PASS | Health API | HTTP 200
PASS | Home API | HTTP 200
PASS | Candidate Profile API | HTTP 200
PASS | Career Recommendations API | HTTP 200
PASS | Skill Gap API | HTTP 200
PASS | CV Download API | HTTP 200
PASS | Job Matching API | HTTP 200
PASS | Mock Interview API | HTTP 200


In [131]:
response = client.get(
    "/resume/parse/skillbridge_test_resume.pdf"
)

print("Resume Parsing:", response.status_code)

Resume Parsing: 200


In [132]:
response = client.post(
    "/mentor/ask",
    json={
        "question": "Give me three important skills to improve my Data Science career."
    }
)

print("Career Mentor:", response.status_code)

Career Mentor: 200


In [133]:
print("REGISTERED SKILLBRIDGE ENDPOINTS")
print("-" * 55)

for route in client.app.routes:
    if hasattr(route, "methods"):
        methods = ",".join(route.methods)
        print(methods, route.path)

REGISTERED SKILLBRIDGE ENDPOINTS
-------------------------------------------------------
GET,HEAD /openapi.json
GET,HEAD /docs
GET,HEAD /docs/oauth2-redirect
GET,HEAD /redoc
GET /
GET /health


In [134]:
registered_paths = {
    route.path
    for route in client.app.routes
    if hasattr(route, "path")
}

print("REGISTERED SKILLBRIDGE ENDPOINTS")
print("-" * 50)

for path in sorted(registered_paths):
    print("FOUND |", path)

REGISTERED SKILLBRIDGE ENDPOINTS
--------------------------------------------------
FOUND | /
FOUND | /docs
FOUND | /docs/oauth2-redirect
FOUND | /health
FOUND | /openapi.json
FOUND | /redoc


In [135]:
required_routes = [
    "/health",
    "/candidate/profile",
    "/resume/upload",
    "/resume/parse/{filename}",
    "/career/recommendations",
    "/career/skill-gap",
    "/cv/download",
    "/jobs/matches",
    "/interview/report",
    "/mentor/ask"
]

print("\nDAY 13 FINAL VALIDATION")
print("-" * 50)

for route in required_routes:
    status = "FOUND" if route in registered_paths else "MISSING"
    print(status, "|", route)

found_count = sum(
    route in registered_paths for route in required_routes
)

print("-" * 50)
print("Expected endpoints:", len(required_routes))
print("Found endpoints:", found_count)
print("Validation:", "PASSED" if found_count == len(required_routes) else "NEEDS REVIEW")


DAY 13 FINAL VALIDATION
--------------------------------------------------
FOUND | /health
MISSING | /candidate/profile
MISSING | /resume/upload
MISSING | /resume/parse/{filename}
MISSING | /career/recommendations
MISSING | /career/skill-gap
MISSING | /cv/download
MISSING | /jobs/matches
MISSING | /interview/report
MISSING | /mentor/ask
--------------------------------------------------
Expected endpoints: 10
Found endpoints: 1
Validation: NEEDS REVIEW


In [136]:
required_routes = [
    "/health",
    "/candidate/profile",
    "/resume/upload",
    "/resume/parse/{filename}",
    "/career/recommendations",
    "/career/skill-gap",
    "/cv/download",
    "/jobs/matches",
    "/interview/report",
    "/mentor/ask"
]

print("\nDAY 13 FINAL VALIDATION")
print("-" * 50)

for route in required_routes:
    status = "FOUND" if route in registered_paths else "MISSING"
    print(status, "|", route)

found_count = sum(
    route in registered_paths for route in required_routes
)

print("-" * 50)
print("Expected endpoints:", len(required_routes))
print("Found endpoints:", found_count)
print("Validation:", "PASSED" if found_count == len(required_routes) else "NEEDS REVIEW")


DAY 13 FINAL VALIDATION
--------------------------------------------------
FOUND | /health
MISSING | /candidate/profile
MISSING | /resume/upload
MISSING | /resume/parse/{filename}
MISSING | /career/recommendations
MISSING | /career/skill-gap
MISSING | /cv/download
MISSING | /jobs/matches
MISSING | /interview/report
MISSING | /mentor/ask
--------------------------------------------------
Expected endpoints: 10
Found endpoints: 1
Validation: NEEDS REVIEW


In [137]:
openapi_data = client.app.openapi()

registered_paths = set(openapi_data.get("paths", {}).keys())

print("SKILLBRIDGE OPENAPI ROUTES")
print("-" * 50)

for path in sorted(registered_paths):
    print("FOUND |", path)

print("-" * 50)
print("Total registered paths:", len(registered_paths))

SKILLBRIDGE OPENAPI ROUTES
--------------------------------------------------
FOUND | /
FOUND | /candidate/profile
FOUND | /candidate/resumes
FOUND | /career/recommendations
FOUND | /career/skill-gap
FOUND | /cv/download
FOUND | /health
FOUND | /interview/report
FOUND | /jobs/matches
FOUND | /mentor/ask
FOUND | /resume/parse/{filename}
FOUND | /resume/upload
--------------------------------------------------
Total registered paths: 12


In [138]:
api_tests = [
    ("Health", "GET", "/health"),
    ("Candidate Profile", "GET", "/candidate/profile"),
    ("Resume Upload", "POST", "/resume/upload"),
    ("Resume Parsing", "GET", "/resume/parse/skillbridge_test_resume.pdf"),
    ("Career Recommendations", "GET", "/career/recommendations"),
    ("Skill Gap", "GET", "/career/skill-gap"),
    ("CV Download", "GET", "/cv/download"),
    ("Job Matching", "GET", "/jobs/matches"),
    ("Mock Interview", "GET", "/interview/report"),
    ("AI Career Mentor", "POST", "/mentor/ask")
]

print("SKILLBRIDGE API ENDPOINT TEST")
print("-" * 65)

for name, method, endpoint in api_tests:

    try:
        if method == "GET":
            result = client.get(endpoint)
        else:
            result = client.post(
                endpoint,
                json={
                    "question": "What skills should I learn for Data Science?"
                }
            )

        status = "PASS" if result.status_code == 200 else "CHECK"

        print(
            f"{status} | {name} | HTTP {result.status_code}"
        )

    except Exception as error:
        print("ERROR |", name, "|", str(error))

SKILLBRIDGE API ENDPOINT TEST
-----------------------------------------------------------------
PASS | Health | HTTP 200
PASS | Candidate Profile | HTTP 200
CHECK | Resume Upload | HTTP 422
PASS | Resume Parsing | HTTP 200
PASS | Career Recommendations | HTTP 200
PASS | Skill Gap | HTTP 200
PASS | CV Download | HTTP 200
PASS | Job Matching | HTTP 200
PASS | Mock Interview | HTTP 200


PASS | AI Career Mentor | HTTP 200


In [139]:
test_pdf = UPLOAD_PATH / "skillbridge_test_resume.pdf"

with open(test_pdf, "rb") as pdf_file:
    upload_response = client.post(
        "/resume/upload",
        files={
            "file": (
                "skillbridge_test_resume.pdf",
                pdf_file,
                "application/pdf"
            )
        }
    )

print("Status code:", upload_response.status_code)
print("Response:", upload_response.json())

Status code: 200
Response: {'status': 'success', 'original_filename': 'skillbridge_test_resume.pdf', 'saved_filename': '6fc5fd73b2b94fc4ab39f98babf4fd16.pdf', 'size_bytes': 1011, 'message': 'Resume uploaded successfully'}


In [140]:
if upload_response.status_code == 200:

    uploaded_name = upload_response.json()["saved_filename"]
    uploaded_path = UPLOAD_PATH / uploaded_name

    print("File exists:", uploaded_path.exists())
    print("File size:", uploaded_path.stat().st_size, "bytes")
else:
    print("Upload failed:", upload_response.json())

File exists: True
File size: 1011 bytes


In [141]:
print("=" * 55)
print("SKILLBRIDGE DAY 13 FINAL VALIDATION")
print("=" * 55)

successful_endpoints = 9

print("Working endpoints:", successful_endpoints + (
    1 if upload_response.status_code == 200 else 0
), "/ 10")

print("Resume Upload:", upload_response.status_code)

if upload_response.status_code == 200:
    print("DAY 13 STATUS: VALIDATION PASSED")
else:
    print("DAY 13 STATUS: NEEDS REVIEW")

SKILLBRIDGE DAY 13 FINAL VALIDATION
Working endpoints: 10 / 10
Resume Upload: 200
DAY 13 STATUS: VALIDATION PASSED


In [142]:
# Multi-resume API smoke tests
import importlib
import backend.api.routes.candidate
import backend.main

importlib.reload(backend.api.routes.candidate)
importlib.reload(backend.main)
client = TestClient(backend.main.app)

library_response = client.get("/candidate/resumes")
print("Resume Library API:", library_response.status_code)
if library_response.status_code == 200:
    print("Saved resumes:", library_response.json().get("total", 0))
else:
    print(library_response.json())

profile_response = client.get("/candidate/profile")
print("Selected profile API:", profile_response.status_code)
print(profile_response.json())

Resume Library API: 200
Saved resumes: 4
Selected profile API: 200
{'status': 'success', 'resume_id': 'candidate-sample-resume-2', 'candidate': {'personal_details': {'full_name': '', 'email': '', 'phone': '', 'location': '', 'linkedin': '', 'portfolio': '', 'professional_summary': ''}, 'education': [], 'experience': [], 'projects': [], 'skills': ['computer science', 'deep learning', 'machine learning', 'Science', 'SQL'], 'certifications': [], 'languages': []}}
